# Style-Weighted vs Plain Gojira — sample comparison

Lightweight A/B: generate N samples from `gojira_lora_r8` and `gojira_lora_r8_sw`,
classify each, and eyeball the text. Not the full evaluation matrix.
Attribution mean is the metric that matters for SW (perplexity was expectedly worse).

In [ ]:
import json
import pandas as pd

from generation.model import load_tokenizer
from config import RESULTS_DIR, Adapter

# Display-only: both adapters are scored by evaluate.py (run_adapters).
# Run `uv run python evaluate.py` first. Only the tokenizer is needed here
# (for the distinctive-token counts), not the base model or classifier.
tok = load_tokenizer()
CACHE_DIR = RESULTS_DIR / "adapters"

In [ ]:
runs = {}
for ad in [Adapter("Gojira", "lora", 8), Adapter("Gojira", "lora", 8, sw=True)]:
    f = CACHE_DIR / ad.result_relpath
    if not f.exists():
        print(f"missing {ad.name} -- run `uv run python evaluate.py`")
        continue
    hit = json.load(open(f))
    runs[ad.name] = (hit["samples"], pd.DataFrame(hit["df"]))
    df = runs[ad.name][1]
    print(f"{ad.name}: mean Gojira attr {df['Gojira'].mean():.3f}  (std {df['Gojira'].std():.3f})")

N = len(next(iter(runs.values()))[0]) if runs else 0

In [5]:
# Side-by-side text
for i in range(N):
    print(f"\n########## SAMPLE {i + 1} ##########")
    for name in runs:
        print(f"\n----- {name} -----")
        print(runs[name][0][i])


########## SAMPLE 1 ##########

----- gojira_lora_r8 -----
I took a boat across the ocean
And I sail away, sailing home
On an open sea that is so deep
I'm falling into my dreams



And yet we all stand upon solid ground
Nowhere else to go and belong
We live in this world of ours, no other place
No one can tell us what to do
A thousand words at last are spoken
This vision is about to die
But now it comes alive when faced with reality
You will see you cannot face the truth
The end has come for good
Ashes to ashes ...


All along I have been dreaming
Of a different time and place
Some day on higher grounds I will find myself
And I'll overcome all obstacles
Until then I am going strong



It may take me hundreds of years
To discover who I am truly
Yet another step forward to reach the point
Where I will understand what happened
In this life there's nothing left to learn
For now, let me be where I want to be

----- gojira_lora_r8_sw -----
I'm dying
No, I can't get over this anymore
Indispo

In [6]:
# Summary: per-adapter mean attribution across all artists
summary = pd.DataFrame({name: runs[name][1].mean() for name in runs})
summary

,gojira_lora_r8,gojira_lora_r8_sw
Death,0.001744,0.012645
Gojira,0.964806,0.816491
Meshuggah,0.001929,0.002972
Opeth,0.028591,0.003211
Tool,0.002929,0.164680


## Distinctive-token count

Turns "I liked SW more" into a number: count how many of Gojira's top
distinctive tokens (the ones SW up-weights) actually appear in each adapter's
output. If SW lands more, that backs the qualitative preference even though the
attribution classifier scored it lower.

In [ ]:
from config import DATA_DIR
from generation.style_loss import build_style_weights
from evaluation.metrics import distinctive_tokens

train_df = pd.read_csv(DATA_DIR / "train.csv")
w = build_style_weights("Gojira", train_df, tok)  # text_col="clean", artist_col="artist" by default
toks = distinctive_tokens(w, tok, n=40)
print("matching on:", sorted(toks))

In [ ]:
from evaluation.metrics import token_occurrences

for name in runs:
    n = len(runs[name][0])
    h = token_occurrences(runs[name][0], toks)
    print(f"{name:<20} {h} hits over {n} samples  ({h / n:.1f}/sample)")

In [ ]:
from evaluation.metrics import token_types

for name in runs:
    print(f"{name:<20} {token_types(runs[name][0], toks):.1f} distinct tokens/sample")